In [2]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [3]:
df = pd.read_csv("../data/train.csv",sep=";")

In [4]:
df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


In [5]:
cat = df.select_dtypes(include="object")


In [6]:
num = df.select_dtypes(exclude="object")

In [7]:
cat.describe()

,job,marital,education,default,housing,loan,contact,month,poutcome,y
count,45211,45211,45211,45211,45211,45211,45211,45211,45211,45211
unique,12,3,4,2,2,2,3,12,4,2
top,blue-collar,married,secondary,no,yes,no,cellular,may,unknown,no
freq,9732,27214,23202,44396,25130,37967,29285,13766,36959,39922


In [8]:
num.describe()

,age,balance,day,duration,campaign,pdays,previous
count,45211.000000,45211.000000,45211.000000,45211.000000,45211.000000,45211.000000,45211.000000
mean,40.936210,1362.272058,15.806419,258.163080,2.763841,40.197828,0.580323
std,10.618762,3044.765829,8.322476,257.527812,3.098021,100.128746,2.303441
min,18.000000,-8019.000000,1.000000,0.000000,1.000000,-1.000000,0.000000
25%,33.000000,72.000000,8.000000,103.000000,1.000000,-1.000000,0.000000
50%,39.000000,448.000000,16.000000,180.000000,2.000000,-1.000000,0.000000
75%,48.000000,1428.000000,21.000000,319.000000,3.000000,-1.000000,0.000000
max,95.000000,102127.000000,31.000000,4918.000000,63.000000,871.000000,275.000000


In [9]:
df.isna().sum()

age          0
job          0
marital      0
education    0
default      0
balance      0
housing      0
loan         0
contact      0
day          0
month        0
duration     0
campaign     0
pdays        0
previous     0
poutcome     0
y            0
dtype: int64

In [10]:
num

,age,balance,day,duration,campaign,pdays,previous
0,58,2143,5,261,1,-1,0
1,44,29,5,151,1,-1,0
2,33,2,5,76,1,-1,0
3,47,1506,5,92,1,-1,0
4,33,1,5,198,1,-1,0
...,...,...,...,...,...,...,...
45206,51,825,17,977,3,-1,0
45207,71,1729,17,456,2,-1,0
45208,72,5715,17,1127,5,184,3
45209,57,668,17,508,4,-1,0


In [8]:
from sklearn.model_selection import train_test_split

In [9]:
X = df.drop(columns=["y"])
y = df.y

In [136]:
X_train,X_test,y_train,y_test = train_test_split(X,y,train_size=0.8)

In [11]:
X_train.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome
23492,57,management,married,tertiary,no,75,no,no,cellular,28,aug,260,5,-1,0,unknown
43394,86,retired,married,primary,no,5236,no,no,telephone,1,apr,558,2,-1,0,unknown
25304,40,technician,married,secondary,no,7313,yes,no,cellular,18,nov,241,2,182,1,failure
8798,29,blue-collar,single,secondary,no,-37,yes,no,unknown,4,jun,182,1,-1,0,unknown
43527,76,retired,married,secondary,no,820,no,no,telephone,23,apr,263,4,-1,0,unknown


In [15]:
len(X_train)

36168

In [12]:
from sklearn.impute import SimpleImputer  # Missing values 
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import FunctionTransformer
from sklearn.preprocessing import MinMaxScaler

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

In [14]:
# now the problem is negative values in the column 
    # if we log the negative values it is infinite 

# so we need to push the series which contain the negative values in to the functions 
    # df[value] = df[] - df[].min + 1


# now to compare this with the function we do 

# here are doing one trasfomer 

def fun(x):
    min_balance=-8019
    bal = np.log1p(x["balance"] - min_balance + 1))
    dur = np.log1p(x["duration"])
    df_1 = pd.DataFrame({"balance":bal,"duration":dur})
    return df_1

fun(df[["balance","duration"]])
        
# ft1 = FunctionTransformer(fun)
# ft1.fit_transform(df[["balance"]])

,balance,duration
0,9.226607,5.568345
1,8.993427,5.023881
2,8.990068,4.343805
3,9.161885,4.532599
4,8.989943,5.293305
...,...,...
45206,9.087721,6.885510
45207,9.185023,6.124683
45208,9.527775,7.028201
45209,9.069813,6.232448


In [53]:
log_pipline = Pipeline([("log", FunctionTransformer(fun,feature_names_out="one-to-one")),
                        ("scalar",StandardScaler())])

num_pipeline = Pipeline([("scaler", StandardScaler())])

nom_pipeline = Pipeline([("one hot",OneHotEncoder(handle_unknown="ignore",sparse_output=False,drop="first"))])

nom_pipeline_1 = Pipeline([("one hot",OneHotEncoder(handle_unknown="ignore",sparse_output=False,drop="if_binary"))])

ordi_pipeline = Pipeline([("ordianl",OrdinalEncoder(categories=[["unknown",'primary','secondary', 'tertiary']]))])


In [45]:
df.columns

Index(['age', 'job', 'marital', 'education', 'default', 'balance', 'housing',
       'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays',
       'previous', 'poutcome', 'y'],
      dtype='object')

In [54]:
num_col = ['age', 'campaign', 'previous']
log_col = ['balance', 'duration']
binary = ["housing","loan"]
nomi_cols = ['job', 'marital', 'contact', 'poutcome']
ord_col = ["education"]

In [30]:
from sklearn.compose import ColumnTransformer

In [55]:
Preprocessor = ColumnTransformer([("num",num_pipeline,num_col),
                                  ("log_num",log_pipline,log_col),
                                  ("binary",nom_pipeline_1,binary),
                                  ("nomianl",nom_pipeline,nomi_cols),
                                  ("ordinal",ordi_pipeline,ord_col)],
                                 remainder='drop')

In [56]:
Preprocessor

,transformers,"[('num', ...), ('log_num', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,copy,True
,with_mean,True
,with_std,True


In [78]:
X_train_processed = Preprocessor.fit_transform(X_train)
X_test_processed  = Preprocessor.transform(X_test)

In [80]:
feature_names = Preprocessor.get_feature_names_out()

X_train_processed = pd.DataFrame(
    X_train_processed,
    columns=feature_names
)

X_test_processed = pd.DataFrame(
    X_test_processed,
    columns=feature_names
)


In [81]:
X_train_processed.head()

,num__age,num__campaign,num__previous,log_num__balance,log_num__duration,binary__housing_yes,binary__loan_yes,nomianl__job_blue-collar,nomianl__job_entrepreneur,nomianl__job_housemaid,...,nomianl__job_unemployed,nomianl__job_unknown,nomianl__marital_married,nomianl__marital_single,nomianl__contact_telephone,nomianl__contact_unknown,nomianl__poutcome_other,nomianl__poutcome_success,nomianl__poutcome_unknown,ordinal__education
0,1.518315,0.727135,-0.242337,-0.572875,0.428397,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,3.0
1,4.257642,-0.247974,-0.242337,1.788639,1.255834,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0
2,-0.087497,-0.247974,0.177215,2.485601,0.346284,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0
3,-1.126552,-0.573011,-0.242337,-0.639583,0.042687,1.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,2.0
4,3.313046,0.402099,-0.242337,-0.151337,0.440814,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,2.0


In [82]:
X_test_processed.head()

,num__age,num__campaign,num__previous,log_num__balance,log_num__duration,binary__housing_yes,binary__loan_yes,nomianl__job_blue-collar,nomianl__job_entrepreneur,nomianl__job_housemaid,...,nomianl__job_unemployed,nomianl__job_unknown,nomianl__marital_married,nomianl__marital_single,nomianl__contact_telephone,nomianl__contact_unknown,nomianl__poutcome_other,nomianl__poutcome_success,nomianl__poutcome_unknown,ordinal__education
0,1.140477,0.077062,-0.242337,-0.701154,0.679940,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0
1,-0.748714,-0.573011,-0.242337,-0.617443,0.496950,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,3.0
2,0.101422,-0.247974,-0.242337,-0.449096,-0.803262,1.0,1.0,0.0,1.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,3.0
3,-1.693309,-0.573011,1.016318,-0.334623,0.171306,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,2.0
4,-0.087497,-0.573011,-0.242337,2.941320,0.166020,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,2.0


In [60]:
X_train_processed.shape

(36168, 26)

In [61]:
X_train.shape

(36168, 16)

In [62]:
Preprocessor.get_feature_names_out()

array(['num__age', 'num__campaign', 'num__previous', 'log_num__balance',
       'log_num__duration', 'binary__housing_yes', 'binary__loan_yes',
       'nomianl__job_blue-collar', 'nomianl__job_entrepreneur',
       'nomianl__job_housemaid', 'nomianl__job_management',
       'nomianl__job_retired', 'nomianl__job_self-employed',
       'nomianl__job_services', 'nomianl__job_student',
       'nomianl__job_technician', 'nomianl__job_unemployed',
       'nomianl__job_unknown', 'nomianl__marital_married',
       'nomianl__marital_single', 'nomianl__contact_telephone',
       'nomianl__contact_unknown', 'nomianl__poutcome_other',
       'nomianl__poutcome_success', 'nomianl__poutcome_unknown',
       'ordinal__education'], dtype=object)

In [63]:
low_features = [
    'nomianl__job_housemaid',
    'nomianl__job_management',
    'nomianl__job_unknown',
    'nomianl__job_technician',
    'nomianl__job_self-employed',
    'nomianl__job_entrepreneur',
    'nomianl__job_retired',
    'nomianl__job_unemployed',
    'nomianl__job_services'
]

In [83]:
X_train_fs = X_train_processed.drop(columns=low_features)
X_test_fs = X_test_processed.drop(columns=low_features)

In [65]:
X_train_fs.shape

(36168, 17)

In [66]:
X_train_fs.columns

Index(['num__age', 'num__campaign', 'num__previous', 'log_num__balance',
       'log_num__duration', 'binary__housing_yes', 'binary__loan_yes',
       'nomianl__job_blue-collar', 'nomianl__job_student',
       'nomianl__marital_married', 'nomianl__marital_single',
       'nomianl__contact_telephone', 'nomianl__contact_unknown',
       'nomianl__poutcome_other', 'nomianl__poutcome_success',
       'nomianl__poutcome_unknown', 'ordinal__education'],
      dtype='object')

In [119]:
X_test_fs.columns

Index(['num__age', 'num__campaign', 'num__previous', 'log_num__balance',
       'log_num__duration', 'binary__housing_yes', 'binary__loan_yes',
       'nomianl__job_blue-collar', 'nomianl__job_student',
       'nomianl__marital_married', 'nomianl__marital_single',
       'nomianl__contact_telephone', 'nomianl__contact_unknown',
       'nomianl__poutcome_other', 'nomianl__poutcome_success',
       'nomianl__poutcome_unknown', 'ordinal__education'],
      dtype='object')

In [96]:
rename_map = {
    "num__age":"age",
    "ordinal__education":"education",
    "binary__housing_yes":"housing_loan",
    "binary__loan_yes":"personal_loan",
    "num__campaign":"contacts_in_campaign",
    "num__previous":"contacted_in_before_campaing",
    "log_num__balance":"balance_log",
    "log_num__duration":"duration_log",
    "nomianl__job_blue-collar":"job_blue-collar",
    "nomianl__job_student":"job_student",
    "nomianl__marital_married":"marital_married",
    "nomianl__marital_single":"marital_single",
    "nomianl__poutcome_other":"previous_outcome_other",
    "nomianl__poutcome_success":"previous_outcome_success",
    "nomianl__poutcome_unknown":"previous_outcome_unknown",
    "nomianl__contact_telephone":"contact_type_telephone",
    "nomianl__contact_unknown":"contact_type_unknown"
}

X_train_fs.rename(columns=rename_map, inplace=True)


In [120]:
X_test_fs.rename(columns=rename_map, inplace=True)

In [121]:
X_test_fs = X_test_fs[['age', 'education', 'housing_loan', 'personal_loan', 'contacts_in_campaign', 'contacted_in_before_campaing', 'balance_log', 'duration_log', 'job_blue-collar', 'job_student', 'marital_married', 'marital_single', 'previous_outcome_other', 'previous_outcome_success', 'previous_outcome_unknown', 'contact_type_telephone', 'contact_type_unknown']]

In [70]:
import joblib

In [71]:
artifact = joblib.load("../models/bank_marketing_final.pkl")
gb_model = artifact["model"]

In [73]:
pipe = Pipeline([
    ("preprocessor", Preprocessor),
    ("model", gb_model)
])

In [88]:
# pipe.predict(X_test)

In [85]:
gb_model

,loss,'log_loss'
,learning_rate,0.1
,n_estimators,100
,subsample,1.0
,criterion,'friedman_mse'
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_depth,3
,min_impurity_decrease,0.0
,init,None


In [105]:
pred = gb_model.predict(X_train_fs)

In [125]:
from sklearn.metrics import confusion_matrix,accuracy_score,classification_report

In [129]:
y_train = y_train.map({"yes":1,"no":0})
y_test = y_test.map({"yes":1,"no":0})

### Data Created through Note book pipeline 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sns
import pandas as pd
import math

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [173]:
path = "../data/train.csv"
df = pd.read_csv(path,sep=";")

# Standardizing the Column Names
# changing the names of columns to understandable format
df.rename(columns={"y":"target"}, inplace=True)
df.rename(columns={"default":"default_credit"}, inplace=True)
df.rename(columns={"housing":"housing_loan"}, inplace=True)
df.rename(columns={"loan":"personal_loan"}, inplace=True)
df.rename(columns={"contact":"contact_type"}, inplace=True)
df.rename(columns={"poutcome":"previous_outcome"}, inplace=True)
df.rename(columns={"pdays":"no_time_contacted_days_before"}, inplace=True)
df.rename(columns={"duration":"last_call_duration"}, inplace=True)
df.rename(columns={"campaign":"contacts_in_campaign"}, inplace=True)
df.rename(columns={"previous":"contacted_in_before_campaing"}, inplace=True)



# Log trasformation
df["balance_log"] = np.log1p(df["balance"] - df["balance"].min() + 1)
df["duration_log"] = np.log1p(df["last_call_duration"])




x = df.drop(columns=["balance","day","month","last_call_duration","no_time_contacted_days_before","target"])


# Encoding

# Label Encoding
maps = {"no": 0,"yes": 1}

x["default_credit"] = x["default_credit"].map(maps)
x["housing_loan"] = x["housing_loan"].map(maps)
x["personal_loan"] = x["personal_loan"].map(maps)

# ordinal encoding 
education_map = {'unknown':0,'primary':1,'secondary':2,'tertiary':3}
x["education"] = x["education"].map(education_map)

# one hot encoding
one_hot_cols = ["job","marital","previous_outcome","contact_type"]
one_hot_encoded = pd.get_dummies(x[one_hot_cols],dtype=int,drop_first=True)


# Making X and Y Data Values
x = pd.concat([x.drop(columns=one_hot_cols),one_hot_encoded],axis=1)
y = df["target"].map(maps)


# splitting
X_train, X_test, y_train, y_test = train_test_split(x,y,test_size=0.2,random_state=42,stratify=y)

# Feature Scaling 
scaler = StandardScaler()

num_cols = ["age",
            "contacts_in_campaign",
            "contacted_in_before_campaing",
            "balance_log",
            "duration_log"]

X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])



print("X - Training length :",len(X_train))
print("X - Testing length :",len(X_test))
print("y - Training length :",len(y_train))
print("y - Testing length :",len(y_test))


low_features = [
    'job_housemaid',
    'job_management',
    'job_unknown',
    'default_credit',
    'job_technician',
    'job_self-employed',
    'job_entrepreneur',
    'job_retired',
    'job_unemployed',
    'job_services'
]

X_train_fs = X_train.drop(columns=low_features)
X_test_fs = X_test.drop(columns=low_features)

X - Training length : 36168
X - Testing length : 9043
y - Training length : 36168
y - Testing length : 9043


In [174]:
X_train_fs.columns

Index(['age', 'education', 'housing_loan', 'personal_loan',
       'contacts_in_campaign', 'contacted_in_before_campaing', 'balance_log',
       'duration_log', 'job_blue-collar', 'job_student', 'marital_married',
       'marital_single', 'previous_outcome_other', 'previous_outcome_success',
       'previous_outcome_unknown', 'contact_type_telephone',
       'contact_type_unknown'],
      dtype='object')

In [175]:
# new data 
pred = model.predict(X_train_fs)
# confusion_matrix(y_test,pred)
print(
"Accuracy", accuracy_score(y_train,pred),
"Precision", precision_score(y_train,pred),
"Recall", recall_score(y_train,pred),
"F1", f1_score(y_train,pred))

Accuracy 0.8544846272948463 Precision 0.4314376826999734 Recall 0.7674308674072323 F1 0.5523517904227269


### Pipline Data genrate

In [166]:


import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer



# Loading the data
df = pd.read_csv("../data/train.csv",sep=";")

# Splitting the data
X = df.drop(columns=["y"])
y = df.y
    

# splitting
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


def log_transform(df):

    temp = df.copy()

    min_balance = -8019

    temp["balance"] = np.log1p(
        temp["balance"] - min_balance + 1
    )

    temp["duration"] = np.log1p(
        temp["duration"]
    )

    return temp



# Columns for each type of transformation   

log_cols = ["balance", "duration"]
num_cols = [
    "age",
    "campaign",
    "previous"
]

binary_cols = [
    "housing",
    "loan"
]

ordinal_cols = [
    "education"
]

nominal_cols = [
    "job",
    "marital",
    "contact",
    "poutcome"
]




# Pipelines for each type of transformation

log_pipeline = Pipeline([
    (
        "log_transform",
        FunctionTransformer(
            log_transform,
            feature_names_out="one-to-one"
        )
    ),
    (
        "scaler",
        StandardScaler()
    )
])


num_pipeline = Pipeline([
    (
        "scaler",
        StandardScaler()
    )
])


binary_pipeline = Pipeline([
    (
        "binary_encoder",
        OneHotEncoder(
            drop="if_binary",
            sparse_output=False,
            handle_unknown="ignore"
        )
    )
])

ordinal_pipeline = Pipeline([
    (
        "ordinal_encoder",
        OrdinalEncoder(
            categories=[
                [
                    "unknown",
                    "primary",
                    "secondary",
                    "tertiary"
                ]
            ]
        )
    )
])

nominal_pipeline = Pipeline([
    (
        "onehot",
        OneHotEncoder(
            drop="first",
            sparse_output=False,
            handle_unknown="ignore"
        )
    )
])


# COlumn transformer to apply the pipelines to the respective columns

preprocessor = ColumnTransformer([
    (
        "num",
        num_pipeline,
        num_cols
    ),

    (
        "log_num",
        log_pipeline,
        log_cols
    ),

    (
        "binary",
        binary_pipeline,
        binary_cols
    ),

    (
        "ordinal",
        ordinal_pipeline,
        ordinal_cols
    ),

    (
        "nominal",
        nominal_pipeline,
        nominal_cols
    )
])


# Preprocessing the training data

X_train_processed = preprocessor.fit_transform(
    X_train
)

feature_names = (
    preprocessor.get_feature_names_out()
)

X_train_processed = pd.DataFrame(
    X_train_processed,
    columns=feature_names
)

# print(preprocessor.get_feature_names_out())


# Selected Features after preprocessing:
selected_features = [
    'num__age',
    'ordinal__education',
    'binary__housing_yes',
    'binary__loan_yes',
    'num__campaign',
    'num__previous',
    'log_num__balance',
    'log_num__duration',
    'nominal__job_blue-collar',
    'nominal__job_student',
    'nominal__marital_married',
    'nominal__marital_single',
    'nominal__poutcome_other',
    'nominal__poutcome_success',
    'nominal__poutcome_unknown',
    'nominal__contact_telephone',
    'nominal__contact_unknown'
]

X_train_fs = X_train_processed[selected_features]

print(X_train_fs.shape)


# Remaping the column names
rename_map = {
    'num__age':'age',
    'ordinal__education':'education',
    'binary__housing_yes':'housing_loan',
    'binary__loan_yes':'personal_loan',
    'num__campaign':'contacts_in_campaign',
    'num__previous':'contacted_in_before_campaing',
    'log_num__balance':'balance_log',
    'log_num__duration':'duration_log',
    'nominal__job_blue-collar':'job_blue-collar',
    'nominal__job_student':'job_student',
    'nominal__marital_married':'marital_married',
    'nominal__marital_single':'marital_single',
    'nominal__poutcome_other':'previous_outcome_other',
    'nominal__poutcome_success':'previous_outcome_success',
    'nominal__poutcome_unknown':'previous_outcome_unknown',
    'nominal__contact_telephone':'contact_type_telephone',
    'nominal__contact_unknown':'contact_type_unknown'
}

X_train_fs.rename(columns=rename_map, inplace=True)

print(X_train_fs.columns.tolist())

(36168, 17)
['age', 'education', 'housing_loan', 'personal_loan', 'contacts_in_campaign', 'contacted_in_before_campaing', 'balance_log', 'duration_log', 'job_blue-collar', 'job_student', 'marital_married', 'marital_single', 'previous_outcome_other', 'previous_outcome_success', 'previous_outcome_unknown', 'contact_type_telephone', 'contact_type_unknown']


C:\Users\midhu\AppData\Local\Temp\ipykernel_36552\3043903391.py:233: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_train_fs.rename(columns=rename_map, inplace=True)


In [153]:
artifact = joblib.load(
    "../models/bank_marketing_final.pkl"
)

model = artifact["model"]
threshold = artifact["threshold"]
features = artifact["features"]



# Metrics
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,confusion_matrix,log_loss,roc_auc_score

In [169]:
# y_train = y_train.map({"yes":1,"no":0})

In [171]:
# y_train

#### Output of PIpeline data 

In [172]:
# new data 
pred = model.predict(X_train_fs)
# confusion_matrix(y_test,pred)
print(
"Accuracy", accuracy_score(y_train,pred),
"Precision", precision_score(y_train,pred),
"Recall", recall_score(y_train,pred),
"F1", f1_score(y_train,pred))

Accuracy 0.8544846272948463 Precision 0.4314376826999734 Recall 0.7674308674072323 F1 0.5523517904227269


### Custom Pipeline Order 

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import FunctionTransformer
from sklearn.preprocessing import MinMaxScaler

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder


import joblib
from sklearn.metrics import confusion_matrix,accuracy_score,classification_report

In [ ]:
# Loading the data
df = pd.read_csv("../data/train.csv",sep=";")

# Splitting the data
X = df.drop(columns=["y"])
y = df.y


X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

# function for the Log Transformer
def fun(x):
    min_balance=-8019
    bal = np.log1p(x["balance"] - min_balance + 1))
    dur = np.log1p(x["duration"])
    df_1 = pd.DataFrame({"balance":bal,"duration":dur})
    return df_1

# fun(df[["balance","duration"]])


# pipelines 

log_pipline = Pipeline([("log", FunctionTransformer(fun,feature_names_out="one-to-one")),
                        ("scalar",StandardScaler())])
num_pipeline = Pipeline([("scaler", StandardScaler())])
nom_pipeline = Pipeline([("one hot",OneHotEncoder(handle_unknown="ignore",sparse_output=False,drop="first"))])
nom_pipeline_1 = Pipeline([("one hot",OneHotEncoder(handle_unknown="ignore",sparse_output=False,drop="if_binary"))])
ordi_pipeline = Pipeline([("ordianl",OrdinalEncoder(categories=[["unknown",'primary','secondary', 'tertiary']]))])


# Columns 
num_col = ['age', 'campaign', 'previous']
log_col = ['balance', 'duration']
binary = ["housing","loan"]
nomi_cols = ['job', 'marital', 'contact', 'poutcome']
ord_col = ["education"]


# preprocessor

Preprocessor = ColumnTransformer([("num",num_pipeline,num_col),
                                  ("log_num",log_pipline,log_col),
                                  ("binary",nom_pipeline_1,binary),
                                  ("nomianl",nom_pipeline,nomi_cols),
                                  ("ordinal",ordi_pipeline,ord_col)],
                                 remainder='drop')



# Fittting X_train, y_train

# returns only the array
X_train_processed = Preprocessor.fit_transform(X_train)
X_test_processed  = Preprocessor.transform(X_test)


# Feature name for the 

feature_names = Preprocessor.get_feature_names_out()



# convert the data into the dataframe 

X_train_processed = pd.DataFrame(
    X_train_processed,
    columns=feature_names
)

X_test_processed = pd.DataFrame(
    X_test_processed,
    columns=feature_names
)



# loading the model 

artifact = joblib.load("../models/bank_marketing_final.pkl")
gb_model = artifact["model"]


# this pipeline fails with the preprocessing and column oreder and all the diffrent issues
pipe = Pipeline([
    ("preprocessor", Preprocessor),
    ("model", gb_model)
])

# ---------------------------------------

# Model prediction 
pred = gb_model.predict(X_train_fs)

# problem 

# 1. Lower features from the past model 
# 2. Order of the features in the past model 
# 3. Model feature names were diffrent 
# 4. Model Scale and features scaled were diffrent compared to the past model 
# 5. 

# Remove the features 

low_features = [
    'nomianl__job_housemaid',
    'nomianl__job_management',
    'nomianl__job_unknown',
    'nomianl__job_technician',
    'nomianl__job_self-employed',
    'nomianl__job_entrepreneur',
    'nomianl__job_retired',
    'nomianl__job_unemployed',
    'nomianl__job_services'
]


X_train_fs = X_train_processed.drop(columns=low_features)
X_test_fs = X_test_processed.drop(columns=low_features)


# renaming the features in X_train after preprocessed

rename_map = {
    "num__age":"age",
    "ordinal__education":"education",
    "binary__housing_yes":"housing_loan",
    "binary__loan_yes":"personal_loan",
    "num__campaign":"contacts_in_campaign",
    "num__previous":"contacted_in_before_campaing",
    "log_num__balance":"balance_log",
    "log_num__duration":"duration_log",
    "nomianl__job_blue-collar":"job_blue-collar",
    "nomianl__job_student":"job_student",
    "nomianl__marital_married":"marital_married",
    "nomianl__marital_single":"marital_single",
    "nomianl__poutcome_other":"previous_outcome_other",
    "nomianl__poutcome_success":"previous_outcome_success",
    "nomianl__poutcome_unknown":"previous_outcome_unknown",
    "nomianl__contact_telephone":"contact_type_telephone",
    "nomianl__contact_unknown":"contact_type_unknown"
}

X_train_fs.rename(columns=rename_map, inplace=True)
X_test_fs.rename(columns=rename_map, inplace=True)



# ordering the model feature
X_train_fs = X_train_fs[['age', 'education', 'housing_loan', 'personal_loan', 'contacts_in_campaign', 'contacted_in_before_campaing', 'balance_log', 'duration_log', 'job_blue-collar', 'job_student', 'marital_married', 'marital_single', 'previous_outcome_other', 'previous_outcome_success', 'previous_outcome_unknown', 'contact_type_telephone', 'contact_type_unknown']]
X_test_fs = X_test_fs[['age', 'education', 'housing_loan', 'personal_loan', 'contacts_in_campaign', 'contacted_in_before_campaing', 'balance_log', 'duration_log', 'job_blue-collar', 'job_student', 'marital_married', 'marital_single', 'previous_outcome_other', 'previous_outcome_success', 'previous_outcome_unknown', 'contact_type_telephone', 'contact_type_unknown']]

In [1]:
import joblib
from sklearn.utils.validation import check_is_fitted

artifact = joblib.load("../models/bank_marketing_final.pkl")
gb_model = artifact["model"]


In [3]:
print(check_is_fitted(gb_model))

None


In [11]:
proba = final_pipeline.predict_proba(X_test)[:,1]

pred = (proba >= artifact["threshold"]).astype(int)

D:\Dev\Envs\batch468\Lib\site-packages\sklearn\pipeline.py:61: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(


In [15]:
from sklearn.metrics import confusion_matrix
print(confusion_matrix(y_test,pred))

[[7218  767]
 [ 346  712]]
